To add: 
- notebook header
- short introduction
- table of contents

This notebook shows how to train a custom crop type model using private reference data.

First ensure your private dataset is uploaded to the RDM...

## Query RDM for private datasets

Check which private collections exist:

In [ ]:
from worldcereal.rdm_api import RdmInteraction

rdm = RdmInteraction()
collections = rdm.get_collections(include_public=False,
                                  include_private=True)
for collection in collections:
    print('----------')
    print(f'ID: {collection.id}')
    print(f'Title: {collection.title}')
    print(f'Sample count: {collection.feature_count}')


2024-12-10 22:02:35.956 | INFO     | worldcereal.rdm_api.rdm_interaction:get_collections:143 - To access private collections, you need to authenticate.


Visit https://sso.terrascope.be/auth/realms/terrascope/device?user_code=FXHP-CYKH 📋 to authenticate.

✅ Authorized successfully

----------
ID: b916755deabe49dbab5f56d67a1c1b39
Title: testpoints_jeroen_small
Sample count: 2
----------
ID: 8c60f4f3e706486fbeff78643ac6fc51
Title: testpoints_maize
Sample count: 8


Now download all samples for your collection of interest:

In [4]:
collection_id = input("Enter the collection id for which you would like to start extractions: ")

gdf = rdm.download_samples(ref_ids=[collection_id])
gdf.head()

2024-12-10 22:02:55.493 | INFO     | worldcereal.rdm_api.rdm_interaction:download_samples:425 - Querying 1 collections...


,sample_id,ewoc_code,valid_time,quality_score_lc,quality_score_ct,extract,h3_l3_cell,ref_id,geometry
0,8c60f4f3e706486fbeff78643ac6fc511,1101060000,2018-08-01,0,0,1,837a6bfffffffff,8c60f4f3e706486fbeff78643ac6fc51,POINT (34.70124 -0.58584)
1,8c60f4f3e706486fbeff78643ac6fc512,1101060000,2018-08-01,0,0,1,837a6bfffffffff,8c60f4f3e706486fbeff78643ac6fc51,POINT (34.76479 -0.59302)
2,8c60f4f3e706486fbeff78643ac6fc513,1101060000,2018-08-01,0,0,1,837a6bfffffffff,8c60f4f3e706486fbeff78643ac6fc51,POINT (34.73686 -0.59883)
3,8c60f4f3e706486fbeff78643ac6fc514,1101060000,2017-08-01,0,0,1,837a6bfffffffff,8c60f4f3e706486fbeff78643ac6fc51,POINT (34.75358 -0.56559)
4,8c60f4f3e706486fbeff78643ac6fc515,1101060000,2017-08-01,0,0,1,837a6bfffffffff,8c60f4f3e706486fbeff78643ac6fc51,POINT (34.71438 -0.57509)


In [ ]:
from pathlib import Path

# specify extractions folder
outfolder = Path('/vitodata/worldcereal/tmp/jeroen/point_extractions')
outfolder.mkdir(parents=True, exist_ok=True)

In [5]:
# save samples to geoparquet
outfolder_col = outfolder / collection_id
outfolder_col.mkdir(parents=True, exist_ok=True)
df_file = str(outfolder_col / 'dataframe.geoparquet')
gdf.to_parquet(df_file)

Now we fire up point extractions for all downloaded samples.

**Note that extractions should be launched separately for each individual collection_id**

In [ ]:
# TODO: start and end date of extractions is set based on range of valid_time in individual jobs (all samples in same year are grouped together)!!
# Should we group every 4 months instead of per year, to avoid having excessive lengths of time series?

In [ ]:
from worldcereal.stac.constants import ExtractionCollection

# TODO: we need a better way to call the extractions script!
import sys
sys.path.append('/data/users/Private/jeroendegerickx/git/worldcereal/worldcereal-classification/scripts/extractions')
from extract import run_extractions

collection_type = ExtractionCollection('POINT_WORLDCEREAL')
run_extractions(collection_type, outfolder_col, Path(df_file))

Now we gather all extractions and prepare the training dataframe...

In [ ]:
# TODO: build a "query_extractions" function that allows to query extractions based on location and timing
# the user will need to be able to only query private collections, only public ones or both
# we can start from "query_public_extractions" and adapt it to also query private extractions


In [ ]:
# this is a temporary solution which just loads all private extractions

from pathlib import Path
import sys
sys.path.append('/data/users/Private/jeroendegerickx/git/worldcereal/worldcereal-classification/scripts/extractions/point_extractions')
from extract_point_worldcereal import load_point_extractions

extract_df = load_point_extractions(outfolder_col)
print(extract_df.head())


   feature_index  S2-L2A-B02  S2-L2A-B03  S2-L2A-B04  S2-L2A-B05  S2-L2A-B06  \
0              0         340         548         590        1118        2160   
1              1         522         911        1248        1529        2144   
2              2         425         750         855        1400        2236   
3              3         353         621         644        1070        2320   
4              4         316         583         754        1087        2135   

   S2-L2A-B07  S2-L2A-B08  S2-L2A-B11  S2-L2A-B12  ...  index  valid_time  \
0        2617        2740        1976        1229  ...      3  2017-08-01   
1        2288        2397        3047        2267  ...      4  2017-08-01   
2        2565        2721        2688        1911  ...      5  2017-08-01   
3        2713        2701        2249        1390  ...      6  2017-08-01   
4        2564        2354        2258        1498  ...      7  2017-08-01   

   quality_score_ct   ewoc_code       h3_l3_cell        

In [ ]:
# You can inspect the timeseries for a specific point or points:

from pathlib import Path
import sys
sys.path.append('/data/users/Private/jeroendegerickx/git/worldcereal/worldcereal-classification/scripts/extractions/point_extractions')
from extract_point_worldcereal import visualize_timeseries

# By default this generates a plot of NDVI and first point in the dataframe
outfile = outfolder_col / 'timeseries_plot.png'
visualize_timeseries(extract_df, outfile)


In [ ]:
# most of the following code is just there temporarily, in order to be able to use the process_parquet function as it was implemented for phase I extractions
# TODO: process_parquet should be adapted to work with the new format of the extractions

# Convert training data frame to presto-compatible format
import numpy as np
import pandas as pd

# create label columns
extract_df['CROPTYPE_LABEL'] = extract_df['ewoc_code'].values
extract_df['LANDCOVER_LABEL'] = np.zeros(extract_df.shape[0])
extract_df['POTAPOV-LABEL-10m'] = np.zeros(extract_df.shape[0])
extract_df['WORLDCOVER-LABEL-10m'] = np.zeros(extract_df.shape[0])

# create DEM columns
extract_df.rename(columns={'elevation': 'DEM-alt-20m'}, inplace=True)
extract_df.rename(columns={'slope': 'DEM-slo-20m'}, inplace=True)

# rename valid_time to valid_date
extract_df['valid_time'] = pd.to_datetime(extract_df['valid_time'])
extract_df.rename(columns={'valid_time': 'valid_date'}, inplace=True)

# others
extract_df['location_id'] = extract_df['sample_id'].values
extract_df['aez_zoneid'] = np.zeros(extract_df.shape[0])

# here we actually run process_parquet:
from worldcereal.utils.refdata import process_parquet
# Prepare training dataframe
training_df = process_parquet(extract_df)
training_df.head()

2024-12-10 22:03:11.670 | INFO     | worldcereal.utils.refdata:process_parquet:292 - Processing selected samples ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   756    0   756    0     0   8307      0 --:--:-- --:--:-- --:--:--  8400
2024-12-10 22:03:11.963 | INFO     | worldcereal.utils.legend:download_latest_legend_from_artifactory:88 - Downloading latest legend file: WorldCereal_LC_CT_legend_20241210.csv
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 78439  100 78439    0     0   823k      0 --:--:-- --:--:-- --:--:--  823k
2024-12-10 22:03:12.253 | INFO     | worldcereal.utils.refdata:process_parquet:356 - Extracted and processed 8 samples from global database.


,CROPTYPE_LABEL,DEM-alt-20m,DEM-slo-20m,LANDCOVER_LABEL,POTAPOV-LABEL-10m,WORLDCOVER-LABEL-10m,aez_zoneid,end_date,lat,lon,...,OPTICAL-B12-ts19-20m,valid_position,available_timesteps,year,label,label_level1,label_level2,label_level3,label_level4,label_level5
sample_id,,,,,,,,,,,,,,,,,,,,,
8c60f4f3e706486fbeff78643ac6fc511,1101060000,1449,7.0,temporary_crops,0.0,0.0,0.0,2019-05-01,-0.585838,34.701237,...,668.0,10,20,2018,maize,temporary_crops,cereals,maize,maize,maize
8c60f4f3e706486fbeff78643ac6fc517,1101060000,1488,5.0,temporary_crops,0.0,0.0,0.0,2018-05-01,-0.579884,34.746923,...,860.0,10,20,2017,maize,temporary_crops,cereals,maize,maize,maize
8c60f4f3e706486fbeff78643ac6fc518,1101060000,1493,5.0,temporary_crops,0.0,0.0,0.0,2018-05-01,-0.587645,34.745686,...,847.0,10,20,2017,maize,temporary_crops,cereals,maize,maize,maize
8c60f4f3e706486fbeff78643ac6fc513,1101060000,1524,4.0,temporary_crops,0.0,0.0,0.0,2019-05-01,-0.598829,34.736862,...,894.0,10,20,2018,maize,temporary_crops,cereals,maize,maize,maize
8c60f4f3e706486fbeff78643ac6fc515,1101060000,1535,3.0,temporary_crops,0.0,0.0,0.0,2018-05-01,-0.575089,34.714384,...,1199.0,10,20,2017,maize,temporary_crops,cereals,maize,maize,maize


Now the user needs to select which crop types to include and which ones to merge...

In [9]:
from utils import CropTypePicker

croptypepicker = CropTypePicker(training_df, samples_threshold=2)
croptypepicker.display()


2024-12-10 22:03:43.217 | WARNING  | utils:_prepare_hierarchy:227 - Less than 2 classes remained after aggregation with threshold 2. Consider adding more data or lowering the threshold.


In [ ]:
# now apply the crop type picker and generate the new dataframe, only containing the classes of interest
# (contained in attribute "final_class")

# QUESTION: in the previous implementation, all non-selected classes were assigned to other_temporary_crops.
# Should we keep this or just remove the non-selected classes from the dataframe?
# currently the latter has been implemented

new_df = croptypepicker.apply()
new_df['final_class'].value_counts()

2024-12-10 22:03:59.567 | INFO     | utils:apply:344 - Discarded 0 samples that do not match the selected crop types.
2024-12-10 22:03:59.569 | INFO     | utils:apply:347 - Selected types: ['maize']


final_class
maize    8
Name: count, dtype: int64

In [ ]:
# Prepare training features

from utils import prepare_training_dataframe

training_dataframe = prepare_training_dataframe(new_df, task_type="croptype")
training_dataframe.head()

In [ ]:
# Train model

from utils import train_classifier

custom_model, report, confusion_matrix = train_classifier(training_dataframe, balance_classes=False)
# Print the classification report
print(report)

In [ ]:
# deploy model

from worldcereal.utils.upload import deploy_model
from openeo_gfmap.backend import cdse_connection
from utils import get_input

modelname = get_input("model")
model_url = deploy_model(cdse_connection(), custom_model, pattern=modelname)
print(f"Your model can be downloaded from: {model_url}")

In [ ]:
# Run inference and visualize output...